# Ejercicio 06 · VAE, evaluación y Model Card

Consolida la evidencia de LAB10 en un producto ejecutable: autoencoder (AE) y autoencoder variacional
(VAE) sobre Fashion-MNIST, con reconstrucción, muestreo, interpolación, coste y límites.

**Pregunta.** ¿Qué se gana y qué se pierde al pasar de un autoencoder determinista a un VAE con
espacio latente de 2 dimensiones, y qué puede y qué no puede afirmarse sobre las muestras
generadas?

**Datos y licencia.** Fashion-MNIST (Zalando, MIT), 60.000 imágenes de entrenamiento y 10.000 de
prueba en escala de grises de 28×28. No contiene datos personales; **no se redistribuye**: se
descarga con `tf.keras.datasets` y no se versiona.

**Alcance.** Es un benchmark didáctico: los resultados describen el comportamiento del modelo en ese
dataset, no en imágenes médicas, de vigilancia ni de documentos.

In [1]:
import json
import os
import sys
import time
from pathlib import Path

# Reproducibilidad bit a bit: un solo hilo por operacion. Con varios hilos,
# oneDNN cambia el orden de las sumas y el VAE entrenado varia entre corridas.
os.environ["TF_DETERMINISTIC_OPS"] = "1"
os.environ["TF_NUM_INTRAOP_THREADS"] = "1"
os.environ["TF_NUM_INTEROP_THREADS"] = "1"

import numpy as np
import tensorflow as tf

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

from inf8239_u03_gen.data import normalize_images, validate_images
from inf8239_u03_gen.figures import guardar_curvas, guardar_interpolacion, guardar_panel
from inf8239_u03_gen.metrics import (
    build_report,
    flatten,
    mean_nearest_neighbor_distance,
    mean_pairwise_distance,
)
from inf8239_u03_gen.models import build_autoencoder, build_vae

SEED = 42
tf.keras.utils.set_random_seed(SEED)
tf.config.experimental.enable_op_determinism()
REPORTS = ROOT / "reports"
MODELS = ROOT / "models"
REPORTS.mkdir(exist_ok=True)
MODELS.mkdir(exist_ok=True)
print("Python", sys.version.split()[0], "| TensorFlow", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

I0000 00:00:1793937143.805887  370860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793937143.806249  370860 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1793937143.835747  370860 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1793937144.668066  370860 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1793937144.668306  370860 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


Python 3.12.15 | TensorFlow 2.21.0 | GPU: []


E0000 00:00:1793937145.714901  370860 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


## 1. Datos y preprocesamiento

Las imágenes se normalizan a [0,1] y la partición de prueba (10.000) se reserva para la evaluación
final. El núcleo del panel es una **muestra aleatoria con semilla fija**: no se eligen los mejores
casos a mano.

In [2]:
(x_train_raw, y_train), (x_test_raw, y_test) = tf.keras.datasets.fashion_mnist.load_data()
x_train = normalize_images(x_train_raw)
x_test = normalize_images(x_test_raw)
validate_images(x_train[:100])
validate_images(x_test[:100])
print("Entrenamiento:", x_train.shape, "| Prueba:", x_test.shape)
print("Clases:", np.unique(y_train, return_counts=True))

rng = np.random.default_rng(SEED)
indice = rng.choice(len(x_test), 1000, replace=False)
x_eval = x_test[indice]
panel_index = np.arange(16)          # posiciones dentro del subconjunto de evaluación
print("Submuestra de evaluación:", x_eval.shape, "| panel:", len(panel_index))
print("Ejemplos de prueba elegidos para el panel:", indice[:16].tolist())

Entrenamiento: (60000, 28, 28, 1) | Prueba: (10000, 28, 28, 1)
Clases: (array([0, 1, 2, 3, 4, 5, 6, 7, 8, 9], dtype=uint8), array([6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000, 6000]))
Submuestra de evaluación: (1000, 28, 28, 1) | panel: 16
Ejemplos de prueba elegidos para el panel: [7006, 7326, 4517, 6283, 3655, 2619, 8318, 1122, 2161, 3970, 6181, 8796, 1370, 2961, 7172, 5483]


## 2. Autoencoder

Cuello de botella de 16 unidades: comprime la imagen a un vector y la reconstruye. Se entrena con
`EarlyStopping(restore_best_weights=True)`, de modo que se conserva el mejor punto y no el último.

In [3]:
autoencoder = build_autoencoder(tf)
autoencoder.compile(optimizer="adam", loss="binary_crossentropy")
inicio = time.perf_counter()
historia_ae = autoencoder.fit(x_train, x_train, validation_split=0.1, epochs=6, batch_size=256,
                              callbacks=[tf.keras.callbacks.EarlyStopping(patience=2, restore_best_weights=True)],
                              verbose=0)
segundos_ae = time.perf_counter() - inicio
perdida_final_ae = historia_ae.history["loss"][-1]
validacion_final_ae = historia_ae.history["val_loss"][-1]
print(f"AE entrenado en {segundos_ae:.2f} s | pérdida final: {perdida_final_ae:.4f} | validación: {validacion_final_ae:.4f}")

reconstruccion_ae = autoencoder.predict(x_eval, verbose=0)
mse_ae = float(np.mean((x_eval - reconstruccion_ae) ** 2))
print(f"MSE de reconstrucción (1.000 ejemplos de prueba): {mse_ae:.5f}")

E0000 00:00:1793937146.283873  370860 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_15}}


AE entrenado en 6.63 s | pérdida final: 0.3060 | validación: 0.3071
MSE de reconstrucción (1.000 ejemplos de prueba): 0.02048


E0000 00:00:1793937152.711395  370860 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


## 3. VAE

La pérdida total es la reconstrucción más la divergencia KL. El espacio latente de 2 dimensiones
permite visualizar, pero **sus ejes no tienen significado interpretable**: no son "tipo de prenda"
ni "color".

La clase `VAE` (muestreo, reparametrización `z = media + exp(0.5 · log_var) · ε`, pérdidas y
`train_step`) está en `src/inf8239_u03_gen/models.py`: el notebook y `scripts/train_ae_vae.py`
importan la misma implementación, sin copias divergentes.

In [4]:
vae = build_vae(tf, latent_dim=2)
encoder, decoder = vae.encoder, vae.decoder
vae.compile(optimizer="adam")
inicio = time.perf_counter()
historia_vae = vae.fit(x_train, epochs=8, batch_size=256, verbose=0)
segundos_vae = time.perf_counter() - inicio
print(f"VAE entrenado en {segundos_vae:.2f} s")

mean, log_variance = encoder.predict(x_eval, verbose=0)
reconstruccion_vae = decoder.predict(mean, verbose=0)
mse_vae = float(np.mean((x_eval - reconstruccion_vae) ** 2))
kl = float(-0.5 * np.mean(np.sum(1 + log_variance - np.square(mean) - np.exp(log_variance), axis=1)))
print(f"MSE de reconstrucción (VAE): {mse_vae:.5f} | divergencia KL media: {kl:.4f}")

VAE entrenado en 13.50 s


E0000 00:00:1793937166.366319  370860 node_def_util.cc:682] NodeDef mentions attribute use_unbounded_threadpool which is not in the op definition: Op<name=MapDataset; signature=input_dataset:variant, other_arguments: -> handle:variant; attr=f:func; attr=Targuments:list(type),min=0; attr=output_types:list(type),min=1; attr=output_shapes:list(shape),min=1; attr=use_inter_op_parallelism:bool,default=true; attr=preserve_cardinality:bool,default=false; attr=force_synchronous:bool,default=false; attr=metadata:string,default=""> This may be expected if your graph generating binary is newer  than this binary. Unknown attributes will be ignored. NodeDef: {{node ParallelMapDatasetV2/_14}}


MSE de reconstrucción (VAE): 0.03361 | divergencia KL media: 6.3072


## 4. Muestreo e interpolación

Una imagen generada **no** es automáticamente privada, original ni libre de sesgo: el análisis de
vecinos cercanos de esta entrega se calcula contra el **conjunto de prueba**, no contra el conjunto
de entrenamiento, de modo que **no permite descartar la memorización**.

In [5]:
muestras = decoder.predict(np.random.default_rng(SEED).normal(size=(16, 2)), verbose=0)
rejilla = np.array([(1 - v) * np.array([-2.0, -1.0]) + v * np.array([2.0, 1.0]) for v in np.linspace(0, 1, 16)])
interpolacion = decoder.predict(rejilla, verbose=0)

originales = x_eval[panel_index]
guardar_panel(REPORTS / "ejercicio06_panel.png", originales, reconstruccion_ae[panel_index],
              reconstruccion_vae[panel_index], muestras)
guardar_interpolacion(REPORTS / "ejercicio06_interpolacion.png", interpolacion)
print("Figuras guardadas: ejercicio06_panel.png, ejercicio06_interpolacion.png")

Figuras guardadas: ejercicio06_panel.png, ejercicio06_interpolacion.png


## 5. Fidelidad, diversidad, memoria y costo

- **Fidelidad**: MSE de reconstrucción sobre 1.000 ejemplos de prueba.
- **Diversidad**: distancia media euclídea entre 64 muestras del decoder, en el espacio de píxeles
  aplanados.
- **Vecinos cercanos**: distancia media de 32 de esas muestras a su vecino más cercano dentro de
  las 1.000 imágenes del **conjunto de prueba** (no contra el conjunto de entrenamiento).
- **Línea base**: distancia media entre pares de esas 1.000 imágenes de prueba; es lo que hace
  interpretable la distancia al vecino más cercano.
- **Memoria**: tamaño de los modelos serializados (`models/*.keras`, no versionados en Git).
- **Costo**: tiempo de entrenamiento e inferencia.

In [6]:
# Diversidad: distancia media euclídea entre 64 muestras del decoder (píxeles aplanados)
latentes = np.random.default_rng(SEED).normal(size=(64, 2))
generadas = decoder.predict(latentes, verbose=0)
plano_generadas = flatten(generadas)
diversidad_muestra = mean_pairwise_distance(plano_generadas)

# Posible memorización: distancia de cada muestra a su vecino más cercano dentro de
# 1.000 ejemplos del conjunto de PRUEBA (no contra el conjunto de entrenamiento)
plano_prueba = flatten(x_eval)
distancia_vecino = mean_nearest_neighbor_distance(plano_generadas[:32], plano_prueba)

# Línea base: distancia media entre pares del propio conjunto de prueba
distancia_pares_prueba = mean_pairwise_distance(plano_prueba)

# Tres puntos latentes fijos: media de píxel de cada imagen decodificada
puntos_latentes = [(0.0, 0.0), (1.5, -1.0), (-2.0, 1.5)]
medias_latentes = [float(decoder.predict(np.array([punto]), verbose=0).mean()) for punto in puntos_latentes]

autoencoder.save(MODELS / "autoencoder.keras")
decoder.save(MODELS / "decoder.keras")
tamano_ae = (MODELS / "autoencoder.keras").stat().st_size / 1024
tamano_decoder = (MODELS / "decoder.keras").stat().st_size / 1024

inicio = time.perf_counter()
_ = decoder.predict(np.zeros((100, 2)), verbose=0)
ms_inferencia = (time.perf_counter() - inicio) * 1000 / 100

metricas = build_report({
    "mse_reconstruccion_ae": mse_ae,
    "mse_reconstruccion_vae": mse_vae,
    "kl_media_vae": kl,
    "medias_decodificacion_latentes": medias_latentes,
    "diversidad_media_entre_muestras": diversidad_muestra,
    "distancia_media_al_vecino_mas_cercano": distancia_vecino,
    "distancia_media_entre_pares_de_prueba": distancia_pares_prueba,
    "segundos_entrenamiento_ae": segundos_ae,
    "segundos_entrenamiento_vae": segundos_vae,
    "ms_inferencia_por_lote_100": ms_inferencia,
    "kb_modelo_ae": tamano_ae,
    "kb_decoder": tamano_decoder,
    "parametros_ae": autoencoder.count_params(),
    "parametros_decoder": decoder.count_params(),
    "ae_final_train_loss": float(historia_ae.history["loss"][-1]),
    "ae_final_val_loss": float(historia_ae.history["val_loss"][-1]),
    "vae_final_reconstruction": float(historia_vae.history["reconstruction"][-1]),
    "vae_final_kl": float(historia_vae.history["kl"][-1]),
    "vae_final_total": float(historia_vae.history["loss"][-1]),
    "seed": SEED,
    "evaluation_examples": len(indice),
    "panel_examples": len(panel_index),
}, origen="notebooks/01_ejercicio06_generativos.ipynb")
(REPORTS / "ejercicio06_metricas.json").write_text(json.dumps(metricas, indent=2), encoding="utf-8")
for clave, valor in metricas.items():
    if isinstance(valor, str):
        texto = valor
    elif isinstance(valor, list):
        texto = ", ".join(f"{numero:.4f}" for numero in valor)
    else:
        texto = str(round(valor, 5))
    print(f"{clave:42s}: {texto}")

origen                                    : notebooks/01_ejercicio06_generativos.ipynb
fecha_ejecucion                           : 2026-11-05T22:52:47-05:00
mse_reconstruccion_ae                     : 0.02048
mse_reconstruccion_vae                    : 0.03361
kl_media_vae                              : 6.30721
medias_decodificacion_latentes            : 0.1041, 0.0533, 0.5050
diversidad_media_entre_muestras           : 4.87691
distancia_media_al_vecino_mas_cercano     : 3.3011
distancia_media_entre_pares_de_prueba     : 11.3984
segundos_entrenamiento_ae                 : 6.63374
segundos_entrenamiento_vae                : 13.50169
ms_inferencia_por_lote_100                : 0.49498
kb_modelo_ae                              : 330.66699
kb_decoder                                : 415.66016
parametros_ae                             : 25888
parametros_decoder                        : 101520
ae_final_train_loss                       : 0.30604
ae_final_val_loss                         : 0.3

## 6. Curvas de entrenamiento

La lectura clave del manual: si la pérdida de entrenamiento baja y la de validación sube, hay
sobreajuste; si ambas quedan altas, subajuste o problema de datos.

In [7]:
guardar_curvas(REPORTS / "ejercicio06_curvas.png", historia_ae.history, historia_vae.history)
print("Curvas guardadas en reports/ejercicio06_curvas.png")

Curvas guardadas en reports/ejercicio06_curvas.png


## 7. Interpretación y decisión

**Observación.** El AE reconstruye mejor (MSE 0,0205) que el VAE (0,0336): la penalización KL
compra un espacio muestreable a cambio de nitidez. La divergencia KL media (6,31) muestra que el
latente se aleja de la referencia normal, y con solo 2 dimensiones las proyecciones se agrupan y las
interpolaciones no siempre son plausibles.

**Evidencia.** Los paneles están construidos sobre una muestra con semilla fija (16 de 1.000 casos de
prueba), no sobre los mejores casos. La diversidad se cuantifica: la distancia media entre 64
muestras del decoder es **4,8769** (distancia euclídea media en el espacio de píxeles aplanados) y
la distancia media de 32 muestras a su vecino más cercano es **3,3011**, calculada **contra 1.000
imágenes del conjunto de prueba** y no contra el conjunto de entrenamiento. La línea base del propio
conjunto de prueba (distancia media entre pares) es **11,3984**, y es lo que hace
interpretable el 3,3011.

**Decisión.** El VAE se acepta como generador ilustrativo del ejercicio y **no** como fuente de
imágenes utilizables: el análisis de vecinos cercanos se hizo contra el conjunto de prueba, así que
**no permite descartar la memorización** y no es concluyente como indicador; el benchmark 28×28 en
grises no es extrapolable a otros dominios. El AE se prefiere cuando el objetivo es reconstruir.
Ambos modelos quedan descritos en la Model Card del repositorio, con sus límites y su costo.